# 👥 Multi-Agent Systems vs. a Monolithic Agent

This notebook introduces the **Multi-Agent System**. This pattern moves beyond the concept of a single agent, no matter how complex, and instead models a team of specialized agents that collaborate to solve a problem. Each agent has a distinct role, persona, and set of skills, mirroring how human expert teams work.

This approach allows for a profound 'division of labor', where complex problems are broken down into sub-tasks and assigned to the agent best suited for the job. To showcase its power, we will conduct a direct comparison. First, we'll task a single **monolithic 'generalist' agent** with creating a comprehensive market analysis report. Then, we will assemble a **specialist team**—a Technical Analyst, a News Analyst, and a Financial Analyst—and have a fourth 'Manager' agent synthesize their expert inputs into a final report. Then we compare the two reports — with code-computed measures and a judge that does not know which approach wrote which report.

## Learning Objectives
In this notebook, you will learn:
1. **A monolithic baseline** - one ReAct agent with web search asked to cover three domains at once
2. **Specialist agents** - three ReAct agents, each with its own persona and the same search tool
3. **Fan-out / fan-in** - run the specialists in parallel and let a manager synthesise their sections
4. **A fair comparison** - the same request, measured in code and scored by a judge that sees reports, not authors

## Where this fits
The first notebook of the module: *why* split work across agents. `02_multiagent_pattern_from_scratch.ipynb`
then shows *how* a crew of agents is built underneath a framework.

## Prerequisites
- Tool-calling / ReAct agents (repository 06 — agent fundamentals)
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` and `TAVILY_API_KEY` in the project-root `.env`


| Property | Value |
|---|---|
| Origin | Fan-out / fan-in (parallel specialists, then one synthesiser) in LangGraph; conceptual ancestors: AutoGen (Microsoft, 2023), CrewAI (2023) |

### Definition
A **Multi-Agent System** is an architecture where a group of distinct, specialized agents collaborate (or sometimes compete) to achieve a common goal. A central controller or a defined workflow protocol is used to manage communication and route tasks between the agents.

### High-level Workflow

1.  **Decomposition:** A main controller or the user provides a complex task.
2.  **Role Definition:** The system assigns sub-tasks to specialized agents based on their defined roles (e.g., 'Researcher', 'Coder', 'Critic', 'Writer').
3.  **Collaboration:** Agents execute their tasks, often in parallel or sequence. They pass their outputs to each other or to a central 'blackboard'.
4.  **Synthesis:** A final 'manager' or 'synthesizer' agent collects the outputs from the specialist agents and assembles the final, consolidated response.

### When to Use / Applications
*   **Complex Report Generation:** Creating detailed reports that require expertise from multiple domains (e.g., financial analysis, scientific research).
*   **Software Development Pipelines:** Simulating a dev team with a programmer, a code reviewer, a tester, and a project manager.
*   **Creative Brainstorming:** A team of agents with different 'personalities' (e.g., one optimistic, one cautious, one wildly creative) can generate a more diverse set of ideas.

### Strengths & Weaknesses
*   **Strengths:**
    *   **Specialization & Depth:** Each agent gets its own persona (a system prompt) and its own tools — no model fine-tuning involved — which leads to higher-quality work in its domain.
    *   **Modularity & Scalability:** It's easy to add, remove, or upgrade individual agents without redesigning the entire system.
    *   **Parallelism:** Multiple agents can work on their sub-tasks simultaneously, potentially reducing overall task time.
*   **Weaknesses:**
    *   **Coordination Overhead:** Managing the communication and workflow between agents adds complexity to the system design.
    *   **Increased Cost & Latency:** Running multiple agents involves more LLM calls, which can be more expensive and slower than a single-agent approach.

---
## 📦 Step 1: Environment Setup

The setup cell loads `OPENAI_API_KEY` and `TAVILY_API_KEY` from the project-root `.env`.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY and TAVILY_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: every agent in this notebook
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini")

print(f"✅ LLM initialized: {llm.model_name}")

---
## 🧍 Part 1: The Baseline — A Monolithic 'Generalist' Agent

To showcase the value of a specialist team, we first need to see how a single agent performs on a complex task. We'll build a ReAct agent (with `langchain.agents.create_agent`) and give it a broad prompt asking it to perform multiple types of analysis at once.

### 1.1 🏗️ Building the Monolithic Agent

**What we are going to do:**
We will construct a standard ReAct agent with `create_agent`. We'll provide it with a web search tool and a very general system prompt that asks it to be a comprehensive financial analyst.

In [ ]:
# ============================================================================
# MONOLITHIC AGENT: ONE REACT AGENT WITH WEB SEARCH
# ============================================================================
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain_tavily import TavilySearch
from rich.console import Console
from rich.markdown import Markdown

console = Console()
search_tool = TavilySearch(max_results=3, name="web_search")
AGENT_CONFIG = {"recursion_limit": 12}  # caps each ReAct loop at a handful of tool rounds

monolithic_agent_app = create_agent(
    llm,
    tools=[search_tool],
    system_prompt="You are a single, expert financial analyst. You must create a comprehensive report covering all aspects of the user's request.",
)
print("✅ Monolithic 'generalist' agent ready")

In [ ]:
monolithic_agent_app

### 1.2 ▶️ Testing the Monolithic Agent

**What we are going to do:**
We'll give the generalist agent a complex task: create a full market analysis report for a company, covering three distinct areas.

In [ ]:
# ============================================================================
# MONOLITHIC AGENT: RUN
# ============================================================================
company = "NVIDIA (NVDA)"
report_query = (
    f"Create a brief but comprehensive market analysis report for {company}. The report should include three "
    "sections: 1. A summary of recent news and market sentiment. 2. A basic technical analysis of the stock's price "
    "trend. 3. A look at the company's recent financial performance."
)

console.print(f"[bold yellow]Testing MONOLITHIC agent:[/bold yellow]\n'{report_query}'\n")
final_mono_output = monolithic_agent_app.invoke({"messages": [HumanMessage(content=report_query)]}, config=AGENT_CONFIG)
mono_report = final_mono_output["messages"][-1].content
searches = sum(len(getattr(m, "tool_calls", []) or []) for m in final_mono_output["messages"])
console.print(f"\n--- [bold red]Monolithic report[/bold red] ({searches} web searches) ---")
console.print(Markdown(mono_report))

**Discussion of the Output:**
The monolithic agent produced a report. The printout above shows how many web searches it made and did its best to synthesize the information. However, the output may have some weaknesses:
- **Lack of Structure:** The sections might blend together, without clear headings or a professional format.
- **Superficial Analysis:** Trying to be an expert in three domains at once, the agent might provide only high-level summaries without much depth in any single area.
- **Generic Tone:** The language might be generic, lacking the specific jargon and focus of a true specialist in each field.

This result is our baseline. It's functional, but not exceptional. Now, we will build a specialist team to see if we can do better.

---
## 👥 Part 2: The Advanced Approach — A Multi-Agent Specialist Team

Now we'll build our team: a News Analyst, a Technical Analyst, and a Financial Analyst. Each will be its own agent node with a specific persona. A final Report Writer will act as the manager, compiling their work.

### 2.1 🧑‍💼 Defining the Specialist Agent Nodes

**What we are going to do:**
We will create three distinct ReAct agents, each with the web search tool. The key difference is the highly specific system prompt we give each one. This prompt defines their persona, their area of expertise, and the exact format their output should take. This is how we enforce specialization.

In [ ]:
# ============================================================================
# SPECIALIST TEAM: THREE REACT AGENTS AND A REPORT WRITER
# ============================================================================
from typing import Optional, TypedDict


class MultiAgentState(TypedDict):
    user_request: str
    news_report: Optional[str]
    technical_report: Optional[str]
    financial_report: Optional[str]
    final_report: Optional[str]


def create_specialist_node(persona: str, output_key: str):
    """Each specialist is a full ReAct agent: it can actually search, not just ask to."""
    agent = create_agent(
        llm,
        tools=[search_tool],
        system_prompt=persona + "\n\nUse web search for current facts. Your output MUST be a concise report section, "
        "formatted in markdown, focusing only on your area of expertise.",
    )

    def specialist_node(state: MultiAgentState) -> dict:
        console.print(f"--- CALLING {output_key.replace('_report', '').upper()} ANALYST ---")
        result = agent.invoke({"messages": [HumanMessage(content=state["user_request"])]}, config=AGENT_CONFIG)
        return {output_key: result["messages"][-1].content}

    return specialist_node


news_analyst_node = create_specialist_node(
    "You are an expert News Analyst. You scour the web for the latest news, articles and sentiment about a company.", "news_report")
technical_analyst_node = create_specialist_node(
    "You are an expert Technical Analyst. You analyse stock price trends and technical indicators.", "technical_report")
financial_analyst_node = create_specialist_node(
    "You are an expert Financial Analyst. You interpret financial statements and performance metrics.", "financial_report")


def report_writer_node(state: MultiAgentState) -> dict:
    """The manager: combine the three sections into one report."""
    console.print("--- CALLING REPORT WRITER ---")
    prompt = f"""You are an expert financial editor. Combine the following specialist reports into a single, professional,
cohesive market analysis report with a brief introduction and conclusion.

News & Sentiment Report:
{state['news_report']}

Technical Analysis Report:
{state['technical_report']}

Financial Performance Report:
{state['financial_report']}
"""
    return {"final_report": llm.invoke(prompt).content}


print("✅ Specialist agents and report writer defined")

### 2.2 🔀 Building the Multi-Agent Graph

The specialists work independently, so they run **in parallel**: `START` fans out to all three, and the report
writer runs once all three have finished (LangGraph waits for every branch of a super-step before the next one).

In [ ]:
# ============================================================================
# SPECIALIST TEAM: FAN-OUT TO THREE ANALYSTS, FAN-IN TO THE WRITER
# ============================================================================
from langgraph.graph import END, START, StateGraph

builder = StateGraph(MultiAgentState)
builder.add_node("news_analyst", news_analyst_node)
builder.add_node("technical_analyst", technical_analyst_node)
builder.add_node("financial_analyst", financial_analyst_node)
builder.add_node("report_writer", report_writer_node)

for analyst in ("news_analyst", "technical_analyst", "financial_analyst"):
    builder.add_edge(START, analyst)          # fan out
    builder.add_edge(analyst, "report_writer")  # fan in
builder.add_edge("report_writer", END)

multi_agent_app = builder.compile()
for edge in multi_agent_app.get_graph().edges:
    print(f"  {edge.source} -> {edge.target}")

---
## ⚔️ Part 3: Head-to-Head Comparison

Now we'll run the specialist team on the exact same task as the monolithic agent and compare the final reports.

In [ ]:
# ============================================================================
# SPECIALIST TEAM: RUN ON THE SAME REQUEST
# ============================================================================
console.print(f"[bold green]Testing MULTI-AGENT TEAM on the same request[/bold green]\n")
final_multi_agent_output = multi_agent_app.invoke({"user_request": report_query})
multi_report = final_multi_agent_output["final_report"]

console.print("\n--- [bold green]Multi-agent report[/bold green] ---")
console.print(Markdown(multi_report))

**What to look for:** does each report cover all three requested sections, with specific figures rather than
generic statements? Each specialist had a single focus and its own search budget, which is where a team usually
gains depth — but it also costs roughly four times the LLM calls. The next part measures the difference instead
of asserting it.

---
## 📏 Part 4: Measured Comparison

Two kinds of evidence:
- **Code-computed measures** — word count, markdown section headings, and whether each requested topic is covered
- **An LLM judge** — each report is scored on its own, as "the report", so the judge never knows which approach
  produced it

One run on one company is a single sample: treat a small score gap as noise, and re-run with other companies before
drawing conclusions.

In [ ]:
# ============================================================================
# MEASURE BOTH REPORTS, THEN A BLIND JUDGE
# ============================================================================
import re

from pydantic import BaseModel, Field


def measure(report: str) -> dict:
    text = report.lower()
    return {
        "words": len(report.split()),
        "headings": len(re.findall(r"^#{1,4} ", report, flags=re.MULTILINE)),
        "covers_news": any(w in text for w in ("news", "sentiment")),
        "covers_technical": any(w in text for w in ("moving average", "rsi", "support", "resistance", "trend")),
        "covers_financials": any(w in text for w in ("revenue", "earnings", "margin", "eps")),
    }


class ReportEvaluation(BaseModel):
    """Scores for one financial report."""
    clarity_and_structure_score: int = Field(description="1-10: organisation, structure and clarity.")
    analytical_depth_score: int = Field(description="1-10: depth and specificity of the analysis in each section.")
    completeness_score: int = Field(description="1-10: how well the report addresses every part of the request.")
    justification: str = Field(description="A brief justification for the scores.")


judge_llm = llm.with_structured_output(ReportEvaluation)


def evaluate_report(report: str) -> ReportEvaluation:
    return judge_llm.invoke(
        f"You are an expert judge of financial analysis reports. Score the report below on a 1-10 scale.\n\n"
        f"**Request:**\n{report_query}\n\n**Report:**\n{report}"
    )


for name, report in [("Monolithic", mono_report), ("Multi-agent", multi_report)]:
    evaluation = evaluate_report(report)
    console.print(f"\n[bold]{name}[/bold]  measures: {measure(report)}")
    console.print(evaluation.model_dump())

**Reading the results:** the code measures are facts about the text (did it cover each topic, how structured is it);
the judge's scores are one model's opinion of one sample. Where both point the same way, the difference is worth
taking seriously; where only the judge prefers one report by a point or two, it is not.

---
## 📝 Summary

In this notebook, we learned:

### 1. A monolithic baseline
- One ReAct agent with search, asked to cover news, technicals and financials in one go

### 2. Specialist agents
- Three ReAct agents with narrow personas and the same search tool; each writes one section

### 3. Fan-out / fan-in
- `START` → three analysts in parallel → report writer; LangGraph waits for all branches before the writer runs

### 4. A fair comparison
- Same request for both, measures computed in code, and a judge that never sees which approach wrote a report
- A team usually buys depth and structure with more LLM calls, cost and latency — measure it for your task

### Next Steps
- **`02_multiagent_pattern_from_scratch.ipynb`**: build a crew of cooperating agents with dependencies from scratch